## Agentic Actions

In [0]:
%pip install --upgrade typing_extensions databricks-langchain langchain langchain-core openai --quiet
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import pandas as pd

SILVER_PAIRS_TABLE   = "tabular.dataexpert.smc_stocks_silver_pairs"
GOLD_TABLE = "tabular.dataexpert.smc_stocks_gold"
GOLD_PAIR_SUMMARY = "tabular.dataexpert.smc_stocks_gold_pair_summary"

trade_date = (
    spark.table(GOLD_TABLE)
    .agg(F.max("trade_date"))
    .collect()[0][0]
)

print(f"Agent run date: {trade_date}")

## Tool 1: Get the latest signal snapshot for all tickers

In [0]:
def get_latest_signals(trade_date) -> pd.DataFrame:
    """
    Returns one row per ticker for the most recent trade_date in Gold.
    This is the agent's starting point — a full snapshot of where
    every ticker stands right now.
    """
    gold_df = spark.table(GOLD_TABLE)

    result = (
        gold_df
        .filter(F.col("trade_date") == trade_date)
        .select(
            "ticker",
            "trade_date",
            "day_close",
            "day_return_pct",
            "rsi",
            "signal_regime",
            "volatility_score",
            "volatility_rank",
            "volume_vs_average_ratio",
            "strong_corr_partner_count",
            "best_intraday_corr_partner",
            "best_intraday_corr_score",
        )
        .orderBy("ticker")
        .toPandas()
    )
    return result

#Testing
signals = get_latest_signals(trade_date)
print(f"Total tickers : {len(signals)}")
display(signals)

## Tool 2: Get tickers where regime changed since yesterday

In [0]:
def get_regime_changes(trade_date) -> pd.DataFrame:
    """
    Returns tickers where the signal regime changed on the latest trade date compared to the previous day.
    A regime change is one of the strongest signals — it means the stock crossed a meaningful RSI/volume threshold it wasn't
    at before.
    """""
    gold_df = spark.table(GOLD_TABLE)

    result = (
        gold_df
        .filter(F.col("trade_date") == trade_date)
        .filter(F.col("regime_changed") == True)
        .filter(F.col("prev_regime").isNotNull()) #excludes first day of trading
        .select(
            "ticker",
            "trade_date",
            "prev_regime",
            "signal_regime",
            "prev_rsi",
            "rsi",
            "day_return_pct",
            "volume_vs_average_ratio",
        )
        .orderBy("ticker")
        .toPandas()
    )
    print(f"Regime changes on {trade_date} : {len(result)}")
    return result
    
#Testing
regime_changes = get_regime_changes(trade_date)
display(regime_changes) 

## Tool 3: Get tickers with unusually high volatility today

In [0]:
def get_high_volatility_tickers(trade_date, threshold: float = 0.75) -> pd.DataFrame:
    """
    Returns tickers whose volatility_rank exceeds the threshold on the latest trade date.
    
    volatility_rank is a percentile (0 to 1) computed across all tickers on the same day — so 0.75 means "top 25% most volatile
    tickers today." This is relative to peers, not an absolute number, which makes it much more meaningful than raw volatility score alone.
    
    Default threshold of 0.75 is a reasonable starting point — adjust based on how many alerts you want the agent to generate.
    """
    gold_df = spark.table(GOLD_TABLE)

    result =(
        gold_df
        .filter(F.col("trade_date") == trade_date)
        .filter(F.col("volatility_rank") >= threshold)
        .select(
            "ticker",
            "trade_date",
            "volatility_score",
            "volatility_rank",
            "rsi",
            "signal_regime",
            "day_return_pct",
        )
        .orderBy(F.desc("volatility_rank"))
        .toPandas()
    )

    print(f"Number of tickers above the threshold of {threshold} on {trade_date} : {len(result)}")
    return result

#Testing
high_volatility_tickers = get_high_volatility_tickers(trade_date, threshold = 0.75)
display(high_volatility_tickers)

## Tool 4: Get correlation context for a specific ticker

In [0]:
def get_correlated_pairs(ticker: str, min_corr_consistency_pct: float = 30.0) -> pd.DataFrame:
    """
    Returns all tickers that are structurally correlated with the given ticker, filtered by how consistently they move together
    across all observed days.

    min_consistency_pct = 30 means "show me pairs that were strongly correlated (corr > 0.3) on at least 30% of all trading days."
    This filters out noise — a one-day correlation spike is not a structural relationship.

    The agent uses this to answer: "when I see a signal on AAPL, do its closest peers show the same signal?" If yes, that's a
    cluster move. If no, AAPL is behaving differently from its usual partners — also interesting.
    """
    pairs_df = spark.table(GOLD_PAIR_SUMMARY)

    #Checking both sides
    result = (
        pairs_df
        .filter((F.col("ticker_a") == ticker) | (F.col("ticker_b") == ticker))
        .filter(F.col("corr_consistency_pct") >= min_corr_consistency_pct)
        .withColumn(
            "partner",
            F.when(F.col("ticker_a") == ticker, F.col("ticker_b")).otherwise(F.col("ticker_a"))
        )
        .select(
            "partner",
            "avg_intraday_corr",
            "max_intraday_corr",
            "strong_corr_days",
            "total_days_observed",
            "corr_consistency_pct"
        )
        .orderBy(F.desc("avg_intraday_corr"))
        .toPandas()
    )

    print(f"Strong partners for {ticker} with a minimum correlation consisteny of {min_corr_consistency_pct}% : {len(result)}")
    return result

#Testing
correlated_pairs = get_correlated_pairs(ticker = "AAPL", min_corr_consistency_pct = 30.0)
display(correlated_pairs)

## Tool 5: Detect divergence in a historically correlated pair

In [0]:
def get_divergence_alert(trade_date, min_consistency_pct: float = 30.0, divergence_threshold: float = 0.2) -> pd.DataFrame:
    """
    Finds pairs that normally move together but diverged today.
    
    Logic:
    - Take all pairs with strong historical consistency (>= min_consistency_pct)
    - Compare their intraday_corr TODAY against their historical avg_intraday_corr
    - If today's correlation dropped significantly below the historical average,
      that's a divergence — something broke the usual relationship
    
    divergence_threshold = 0.3 means today's corr is 0.3 lower than the 
    historical average. Adjust based on how sensitive you want the agent to be.
    """
    gold_df    = spark.table(GOLD_TABLE)
    pairs_df   = spark.table(SILVER_PAIRS_TABLE)
    summary_df = spark.table(GOLD_PAIR_SUMMARY)

    # Get today's correlation for all pairs
    today_pairs = (
        pairs_df
        .filter(F.col("trade_date") == trade_date)
        .select("ticker_a", "ticker_b", "intraday_corr")
    )

    # Guard against empty today_pairs before joining
    if today_pairs.limit(1).count() == 0:
        print(f"No pair data found for {trade_date}")
        return pd.DataFrame(columns=[
            "ticker_a", "ticker_b", "historical_avg_corr",
            "today_corr", "corr_drop", "corr_consistency_pct"
        ])

    # Join against historical summary for pairs above consistency threshold
    result = (
        summary_df
        .filter(F.col("corr_consistency_pct") >= min_consistency_pct)
        .join(today_pairs, on=["ticker_a", "ticker_b"], how="left")
        .withColumn(
            "corr_drop",
            F.round(F.col("avg_intraday_corr") - F.col("intraday_corr"), 4)
        )
        .filter(F.col("corr_drop") >= divergence_threshold)
        .select(
            "ticker_a",
            "ticker_b",
            F.col("avg_intraday_corr").alias("historical_avg_corr"),
            F.col("intraday_corr").alias("today_corr"),
            "corr_drop",
            "corr_consistency_pct",
        )
        .orderBy(F.desc("corr_drop"))
        .toPandas()
    )

    print(f"Divergent pairs on {trade_date} (drop >= {divergence_threshold}): {len(result)}")
    return result

# ── Test it ─────────────────────────────────────────────────────
divergences = get_divergence_alert(trade_date, min_consistency_pct=30.0, divergence_threshold=0.2)

if len(divergences) > 0:
    display(divergences)
else:
    print("No divergent pairs found")

## LLM Reasoning

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  
    temperature=0.1,   # low temperature = more consistent, less creative
    max_tokens=2000,
)

# Quick sanity check. invoke meaning - Execute the model (or chain) with this input and give me the result.
response = llm.invoke("Reply with only the words: LLM connected successfully.")
print(response.content)

## Dry Run
Calling all the agent tools upfront, bundling them into a prompt and asking the LLM to reason over it

In [0]:
#Step 1 - Collecting function outputs
signals = get_latest_signals(trade_date)
regime_changes = get_regime_changes(trade_date)
high_volatility_tickers = get_high_volatility_tickers(trade_date, threshold = 0.75)
divergences = get_divergence_alert(trade_date, min_consistency_pct=30.0, divergence_threshold=0.2)

#Step 2 - formatting into a prompt
prompt = f"""
You are a quantitative stock market analyst. You have access to enriched daily stock data for {len(signals)} tickers as of {trade_date}.

Below is a summary of today's market data. Analyze it and identify which tickers warrant a trading signal and why. Be specific and reference the 
actual numbers.

--- FULL SIGNAL SNAPSHOT ---
{signals[["ticker", "rsi", "signal_regime", "volatility_rank", "volume_vs_average_ratio", "best_intraday_corr_partner", "best_intraday_corr_score", "day_return_pct"]].to_string(index = False)}

--- REGIME CHANGES TODAY FOR {len(signals)} TICKERS ---
{regime_changes[["ticker", "prev_regime", "signal_regime", "prev_rsi", "rsi", "day_return_pct"]].to_string(index = False)
if len(regime_changes) > 0 else "None"}

--- TICKERS WITH HIGH VOLATILITY RANKS ---
{high_volatility_tickers[["ticker", "volatility_rank", "rsi", "signal_regime"]].to_string(index = False)
if len(high_volatility_tickers) > 0 else "None"}

--- DIVERGENT TICKER PAIRS TODAY ---
{divergences[["ticker_a", "ticker_b", "historical_avg_corr", "today_corr", "corr_drop"]].to_string(index = False)
if len(divergences) > 0 else "None"}

Based on this data, identify the most significant signals. For each signal:
1. Name the ticker or pair
2. State the signal type (MOMENTUM_ALERT, OVERSOLD_ALERT, REGIME_CHANGE, VOLATILITY_SPIKE, CORRELATION_BREAK, CLUSTER_MOVE)
3. Explain your reasoning using the actual numbers
4. Assign a confidence score between 0.0 and 1.0
"""

# Step 3 — send to LLM
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

## Agent Loop

In [0]:
from langchain_core.tools import tool

@tool
def tool_get_latest_signals(trade_date: str) -> str:
    """Returns a snapshot of all tickers for the given trade date. Includes RSI, signal regime, volatility rank, volume ratio, and best correlated partner for each ticker. Use this as the starting point for every analysis run."""
    df = get_latest_signals(trade_date)
    return df.to_string(index = False) if len(df) > 0 else "No data found"

@tool
def tool_get_regime_changes(trade_date: str) -> str:
    """Returns tickers where the signal regime changed on the given trade date compared to the previous day. Use this to identify tickers with meaningful momentum shifts."""
    df = get_regime_changes(trade_date)
    return df.to_string(index = False) if len(df) > 0 else "No regime changes found"

@tool
def tool_get_high_volatility_tickers(trade_date: str, threshold: float = 0.75) -> str:
    """Returns tickers in the top percentile of volatility on the given trade date. threshold=0.75 means top 25% most volatile. Use this to identify tickers with unusual price movement."""
    df = get_high_volatility_tickers(trade_date, threshold)
    return df.to_string(index = False) if len(df) > 0 else "No high volatility tickers found"

@tool
def tool_get_correlated_pairs(ticker: str, min_consistency_pct: float = 30.0) -> str:
    """
    Returns structural correlation partners for a specific ticker. Use this after identifying an interesting ticker to understand
    whether its correlated peers show the same signal — if yes, that strengthens the signal confidence.
    """
    df = get_correlated_pairs(ticker, min_consistency_pct)
    return df.to_string(index=False) if len(df) > 0 else f"No structural partners found for {ticker}."

@tool
def tool_get_divergence_alert(trade_date: str, min_consistency_pct: float = 30.0, divergence_threshold: float = 0.2) -> str:
    """Returns pairs that historically move together but diverged today. Use this to identify CORRELATION_BREAK signals"""
    df = get_divergence_alert(trade_date, min_consistency_pct, divergence_threshold)
    return df.to_string(index = False) if len(df) > 0 else "No divergent pairs found"

tools = [
    tool_get_latest_signals,
    tool_get_regime_changes,
    tool_get_high_volatility_tickers,
    tool_get_correlated_pairs,
    tool_get_divergence_alert
]

print(f"Successfully registered {len(tools)} tools.")

## Binding tools to the LLM and writing the agent loop

In [0]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
import json

# Step 1 — bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Step 2 — define system prompt
system_prompt = f"""
You are a quantitative stock market analyst monitoring {len(signals)} equity tickers. Today's date is {trade_date}.

You have access to five tools to query enriched stock data:
- tool_get_latest_signals: start here for every analysis
- tool_get_regime_changes: find tickers with momentum shifts
- tool_get_high_volatility_tickers: find unusually volatile tickers
- tool_get_correlated_pairs: get correlation partners for a specific ticker
- tool_get_divergence_alert: find pairs that broke their usual relationship

Your job is to:
1. Call the tools to gather data
2. Reason over the results
3. Identify tickers or pairs that warrant a signal
4. For each signal produce a JSON object with exactly these fields:
   {{
     "ticker": "AAPL",
     "signal_type": "MOMENTUM_ALERT",
     "rationale": "your reasoning here referencing actual numbers",
     "confidence": 0.85,
     "rsi_at_signal": 74.2,
     "regime_at_signal": "STRONG_SELL"
   }}

Valid signal types: MOMENTUM_ALERT, OVERSOLD_ALERT, REGIME_CHANGE, VOLATILITY_SPIKE, CORRELATION_BREAK, CLUSTER_MOVE

Rules:
- Only fire a signal if the data genuinely supports it
- Do not force a signal if nothing notable exists
- Always call tool_get_latest_signals first
- When you find an interesting ticker, call tool_get_correlated_pairs
  to check if peers show the same pattern
- When finished output all signals as a JSON array like this:
  SIGNALS: [{{...}}, {{...}}]
"""

# Step 3 — tool lookup map so we can execute tool calls by name
tool_map = {t.name: t for t in tools}

# Step 4 — agent loop
def run_agent(trade_date: str, max_iterations: int = 10) -> list:
    """
    Runs the agent loop until the LLM stops calling tools and produces its final SIGNALS output.
    """
    messages = [
        SystemMessage(content=system_prompt), #the instruction
        HumanMessage(content=f"Analyze today's stock data for {trade_date} and generate signals.") #user input
    ]

    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")

        # Call LLM
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # Check if LLM wants to call tools
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                print(f"Tool call: {tool_name}({tool_args})")

                # Execute the tool
                tool_fn = tool_map[tool_name]
                tool_result = tool_fn.invoke(tool_args)

                # Append tool result to messages
                messages.append(ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                ))

        else:
            # LLM stopped calling tools — extract final output
            print("\n--- Agent finished reasoning ---")
            print(response.content)
            return parse_signals(response.content)

    print("Max iterations reached.")
    return []

# Step 5 — parse signals from LLM output
def parse_signals(llm_output: str) -> list:
    """
    Extracts the JSON signal array from the LLM final response.
    """
    try:
        start = llm_output.index("SIGNALS:") + len("SIGNALS:")
        json_str = llm_output[start:].strip()
        signals_list = json.loads(json_str)
        print(f"\nParsed {len(signals_list)} signals successfully.")
        return signals_list
    except (ValueError, json.JSONDecodeError) as e:
        print(f"Could not parse signals: {e}")
        print("Raw output:", llm_output)
        return []

# Step 6 — run it
parsed_signals = run_agent(str(trade_date))
print(f"\nFinal signal count: {len(parsed_signals)}")
for s in parsed_signals:
    print(s)

## Writing signals as alerts to table

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, DateType, TimestampType
)
import datetime

ALERTS_TABLE = "tabular.dataexpert.smc_stocks_alerts"

# Step 1 — define alerts schema explicitly
alerts_schema = StructType([
    StructField("ticker",           StringType(),    True),
    StructField("trade_date",       DateType(),      True),
    StructField("signal_type",      StringType(),    True),
    StructField("rationale",        StringType(),    True),
    StructField("confidence",       DoubleType(),    True),
    StructField("rsi_at_signal",    DoubleType(),    True),
    StructField("regime_at_signal", StringType(),    True),
    StructField("agent_run_ts",     TimestampType(), True),
])

# Step 2 — write signals to alerts table
def write_signals(signals_list: list, trade_date):
    if not signals_list:
        print("No signals to write.")
        return

    # Normalize and add metadata to each signal
    rows = []
    for s in signals_list:
        rows.append((
            s.get("ticker"),
            trade_date,
            s.get("signal_type"),
            s.get("rationale"),
            float(s.get("confidence",    0.0)),
            float(s.get("rsi_at_signal", 0.0)),
            s.get("regime_at_signal"),
            datetime.datetime.utcnow(),
        ))

    # Create Spark DataFrame from parsed signals
    signals_df = spark.createDataFrame(rows, schema=alerts_schema)

    # Upsert into alerts table — idempotent on ticker + trade_date + signal_type
    if spark.catalog.tableExists(ALERTS_TABLE):
        delta_table = DeltaTable.forName(spark, ALERTS_TABLE)
        (
            delta_table.alias("target")
            .merge(
                signals_df.alias("source"),
                """
                target.ticker      = source.ticker      AND
                target.trade_date  = source.trade_date  AND
                target.signal_type = source.signal_type
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            signals_df.write
            .format("delta")
            .saveAsTable(ALERTS_TABLE)
        )

    print(f"Written {len(signals_list)} signal(s) to {ALERTS_TABLE}")

# Step 3 — write the parsed signals
write_signals(parsed_signals, trade_date)

# Step 4 — verify
print("\n=== ALERTS TABLE ===")
alerts_df = spark.table(ALERTS_TABLE)
print(f"Total alerts: {alerts_df.count():,}")
display(alerts_df.orderBy("trade_date", "ticker"))

## Agent pipeline

In [0]:
def run_agent_pipeline():
    """
    Full agent pipeline — called by Databricks Workflow after Gold tables are populated. Scans Gold, reasons over the data,
    and writes signals to the alerts table.
    """
    print("=" * 60)
    print(f"Agent pipeline started : {datetime.datetime.now(datetime.UTC)}")
    print("=" * 60)

    # Step 1 — compute latest date
    trade_date = (
        spark.table(GOLD_TABLE)
        .agg(F.max("trade_date"))
        .collect()[0][0]
    )
    print(f"Analyzing trade date : {trade_date}")

    # Step 2 — run agent loop
    parsed_signals = run_agent(str(trade_date))

    # Step 3 — write signals to alerts table
    write_signals(parsed_signals, trade_date)

    # Step 4 — summary
    print("\n" + "=" * 60)
    print(f"Agent pipeline complete")
    print(f"Trade date    : {trade_date}")
    print(f"Signals fired : {len(parsed_signals)}")
    for s in parsed_signals:
        print(f"  {s.get('ticker'):6} | {s.get('signal_type'):20} | confidence: {s.get('confidence')}")
    print("=" * 60)

    return parsed_signals

# Run it
run_agent_pipeline()